In [1]:
import numpy as np
import cvxpy as cp

np.random.seed(42)

# ----------------------------
# 1. SETUP
# ----------------------------
n_assets = 4
n_scenarios = 5000
beta = 0.95
target_return = 0.06  # 6% target return, matching the dual example

# ----------------------------
# 2. CASE 1: ALL NORMAL RETURNS
# ----------------------------
means_normal = np.array([0.08, 0.06, 0.05, 0.07])
std_devs = np.array([0.20, 0.15, 0.10, 0.18])

corr = np.array([
    [1.0, 0.3, 0.1, 0.2],
    [0.3, 1.0, 0.2, 0.1],
    [0.1, 0.2, 1.0, 0.3],
    [0.2, 0.1, 0.3, 1.0],
])
cov_normal = np.outer(std_devs, std_devs) * corr

scenarios_case1 = np.random.multivariate_normal(means_normal, cov_normal, size=n_scenarios)

# ----------------------------
# 3. CASE 2: REPLACE ASSET 4 WITH A CRASH-RISK ASSET
#    Two-point distribution: return = loss_crash with probability p_crash,
#    otherwise gain_good. Chosen so that its mean AND variance equal
#    those of the normal Asset 4.
#
#    With d = loss_crash - mean, the variance of a two-point distribution is
#        var = p * d^2 / (1 - p)   =>   d = -sqrt(var * (1 - p) / p)
# ----------------------------
target_mean = means_normal[3]
target_var = std_devs[3] ** 2

p_crash = 0.10
loss_crash = target_mean - np.sqrt(target_var * (1 - p_crash) / p_crash)   # -0.47
gain_good = (target_mean - p_crash * loss_crash) / (1 - p_crash)           # +0.13

implied_var = (p_crash * (loss_crash - target_mean) ** 2
               + (1 - p_crash) * (gain_good - target_mean) ** 2)

print("=== Asset 4 replacement diagnostics ===")
print(f"Crash return: {loss_crash:.4f} (probability {p_crash:.2f}), good-state return: {gain_good:.4f}")
print(f"Target mean: {target_mean:.4f}, implied mean: {p_crash*loss_crash + (1-p_crash)*gain_good:.4f}")
print(f"Target variance: {target_var:.6f}, implied variance: {implied_var:.6f}\n")

# Assets 1-3 keep the same draws as Case 1; only asset 4 is replaced.
scenarios_case2 = scenarios_case1.copy()
crash_draws = np.random.rand(n_scenarios) < p_crash
scenarios_case2[:, 3] = np.where(crash_draws, loss_crash, gain_good)

print(f"Empirical mean of new Asset 4: {scenarios_case2[:, 3].mean():.4f}")
print(f"Empirical std of new Asset 4:  {scenarios_case2[:, 3].std():.4f} "
      f"(original Asset 4 std: {std_devs[3]:.4f})\n")


# ----------------------------
# 4. MEAN-VARIANCE OPTIMISATION (long-only, target return)
# ----------------------------
def clean_weights(w):
    """Remove tiny negative numbers from the solver and make the weights add up to 1."""
    w = np.maximum(w, 0.0)
    return w / w.sum()


def solve_mean_variance(scenarios, target_return):
    n = scenarios.shape[1]
    mu = scenarios.mean(axis=0)
    Sigma = np.cov(scenarios, rowvar=False)

    x = cp.Variable(n)
    objective = cp.Minimize(0.5 * cp.quad_form(x, cp.psd_wrap(Sigma)))
    constraints = [
        mu @ x >= target_return,
        cp.sum(x) == 1,
        x >= 0,
    ]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    assert prob.status in ("optimal", "optimal_inaccurate"), prob.status
    return clean_weights(x.value)


# ----------------------------
# 5. CVAR OPTIMISATION (long-only, target return): Rockafellar-Uryasev LP
# ----------------------------
def solve_cvar(scenarios, beta, target_return):
    q, n = scenarios.shape
    mu = scenarios.mean(axis=0)

    x = cp.Variable(n)
    alpha = cp.Variable()
    u = cp.Variable(q)

    losses = -scenarios @ x  # loss in scenario k = -(return of the portfolio)

    objective = cp.Minimize(alpha + (1 / (q * (1 - beta))) * cp.sum(u))
    constraints = [
        u >= losses - alpha,
        u >= 0,
        mu @ x >= target_return,
        cp.sum(x) == 1,
        x >= 0,
    ]
    prob = cp.Problem(objective, constraints)
    prob.solve()
    assert prob.status in ("optimal", "optimal_inaccurate"), prob.status
    return clean_weights(x.value)


# ----------------------------
# 6. EVALUATE ANY FIXED PORTFOLIO ON THE SAME SCENARIOS
# ----------------------------
def evaluate(scenarios, x, beta):
    """Average return, VaR and CVaR of a fixed portfolio, measured the same way for every method."""
    returns = scenarios @ x
    losses = -returns
    var = np.quantile(losses, beta)
    cvar = losses[losses >= var].mean()
    return returns.mean(), var, cvar


def compare(label, scenarios):
    x_mv = solve_mean_variance(scenarios, target_return)
    x_cvar = solve_cvar(scenarios, beta, target_return)

    ret_mv, var_mv, cvar_mv = evaluate(scenarios, x_mv, beta)
    ret_cv, var_cv, cvar_cv = evaluate(scenarios, x_cvar, beta)

    print(f"========== {label} ==========")
    print(f"Mean-variance weights: {np.round(x_mv, 4)}")
    print(f"CVaR weights:          {np.round(x_cvar, 4)}")
    print(f"Weight difference (L1 norm): {np.sum(np.abs(x_mv - x_cvar)):.4f}")
    print(f"Weight on Asset 4 -> MV: {x_mv[3]:.4f}, CVaR: {x_cvar[3]:.4f}")
    print(f"Expected return   -> MV: {ret_mv:.4f}, CVaR: {ret_cv:.4f}")
    print(f"VaR  (95%)        -> MV: {var_mv:.4f}, CVaR: {var_cv:.4f}")
    print(f"CVaR (95%)        -> MV: {cvar_mv:.4f}, CVaR: {cvar_cv:.4f}")
    print(f"=> MV portfolio carries {(cvar_mv / cvar_cv - 1) * 100:.1f}% more CVaR "
          f"than the CVaR-optimal portfolio, at the same target return.\n")
    return x_mv, x_cvar


# ----------------------------
# 7. RUN BOTH CASES
# ----------------------------
x_mv_1, x_cvar_1 = compare("CASE 1: ALL NORMAL RETURNS", scenarios_case1)
x_mv_2, x_cvar_2 = compare("CASE 2: CRASH-RISK ASSET 4 (same mean and variance)", scenarios_case2)

=== Asset 4 replacement diagnostics ===
Crash return: -0.4700 (probability 0.10), good-state return: 0.1300
Target mean: 0.0700, implied mean: 0.0700
Target variance: 0.032400, implied variance: 0.032400

Empirical mean of new Asset 4: 0.0695
Empirical std of new Asset 4:  0.1806 (original Asset 4 std: 0.1800)

========== CASE 1: ALL NORMAL RETURNS ==========
Mean-variance weights: [0.2352 0.1731 0.3693 0.2224]
CVaR weights:          [0.2299 0.1801 0.363  0.227 ]
Weight difference (L1 norm): 0.0234
Weight on Asset 4 -> MV: 0.2224, CVaR: 0.2270
Expected return   -> MV: 0.0600, CVaR: 0.0600
VaR  (95%)        -> MV: 0.0980, CVaR: 0.0971
CVaR (95%)        -> MV: 0.1352, CVaR: 0.1351
=> MV portfolio carries 0.0% more CVaR than the CVaR-optimal portfolio, at the same target return.

========== CASE 2: CRASH-RISK ASSET 4 (same mean and variance) ==========
Mean-variance weights: [0.1953 0.1378 0.3991 0.2678]
CVaR weights:          [0.2681 0.1678 0.4018 0.1622]
Weight difference (L1 norm): 0.2